# DE2 — Lab 3: Graphs or Clustering — Instrumented Iterative Workload (15%)
Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026
Track: A — Esports
Path chosen: A — Graph Processing (Hero co-pick graph + iterative PageRank)
Goal: Execute an iterative workload with partitioning strategies, skew analysis, per-iteration shuffle costs, convergence, and a before/after comparative report.

In [1]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType
import time, pathlib, csv, io, contextlib, shutil, json
from datetime import datetime

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")

spark = SparkSession.builder \
    .appName("DE2-Lab3-Iterative") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

# Set checkpoint directory to break lineage during iterations
spark.sparkContext.setCheckpointDir("checkpoint_tmp")

show_spark_ui(spark)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 12:56:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## A.1 — Build Edge and Vertex DataFrames (hero co-pick graph)

In [2]:
# Load match-pick data (one row per (match_id, hero_id))
schema = StructType([
    StructField("match_id", IntegerType(), False),
    StructField("hero_id",  StringType(),  False),
])
df_picks = spark.read.schema(schema).option("header", "true").csv("data/match_picks.csv")
print(f"Picks: {df_picks.count()}")
df_picks.show(5)

# Build edges: for each match, all pairs of distinct heroes (co-pick)
p1 = df_picks.alias("p1")
p2 = df_picks.alias("p2")
edges = (p1.join(p2, (F.col("p1.match_id") == F.col("p2.match_id")) & (F.col("p1.hero_id") < F.col("p2.hero_id")))
    .select(F.col("p1.hero_id").alias("src"), F.col("p2.hero_id").alias("dst"))
    .distinct())

edges.cache()
n_edges = edges.count()
print(f"Edges (unique co-pick pairs): {n_edges}")
edges.show(5)

# Vertices: union of all heroes appearing as src or dst
vertices = (edges.select(F.col("src").alias("id"))
            .union(edges.select(F.col("dst").alias("id")))
            .distinct())
vertices.cache()
n_vertices = vertices.count()
print(f"Vertices: {n_vertices}")

Picks: 1000
+--------+-------+
|match_id|hero_id|
+--------+-------+
|       1|hero_01|
|       1|hero_02|
|       1|hero_12|
|       1|hero_15|
|       1|hero_10|
+--------+-------+
only showing top 5 rows
Edges (unique co-pick pairs): 586
+-------+-------+
|    src|    dst|
+-------+-------+
|hero_01|hero_10|
|hero_12|hero_15|
|hero_04|hero_10|
|hero_07|hero_21|
|hero_01|hero_13|
+-------+-------+
only showing top 5 rows
Vertices: 50


## A.1bis — Skew Analysis (find hot vertices)

In [3]:
# Out-degree distribution = how many neighbors each vertex has
all_endpoints = edges.select(F.col("src").alias("id")).union(edges.select(F.col("dst").alias("id")))
degree = all_endpoints.groupBy("id").count().withColumnRenamed("count", "degree")

print("Top 10 hot vertices (highest degree):")
degree.orderBy(F.desc("degree")).show(10)

print("Degree statistics:")
degree.describe("degree").show()

Top 10 hot vertices (highest degree):
+-------+------+
|     id|degree|
+-------+------+
|hero_01|    49|
|hero_02|    48|
|hero_03|    45|
|hero_04|    45|
|hero_05|    40|
|hero_06|    39|
|hero_09|    38|
|hero_07|    37|
|hero_10|    37|
|hero_11|    34|
+-------+------+
only showing top 10 rows
Degree statistics:
+-------+------------------+
|summary|            degree|
+-------+------------------+
|  count|                50|
|   mean|             23.44|
| stddev|11.021611423351258|
|    min|                 7|
|    max|                49|
+-------+------------------+



## A.2 — Iterative PageRank — RUN 1 (default partitioning)

In [4]:
NUM_ITER = 10
DAMPING = 0.85

# Make edges bidirectional (undirected co-pick graph)
edges_undirected = (edges.select(F.col("src"), F.col("dst"))
                    .union(edges.select(F.col("dst").alias("src"), F.col("src").alias("dst"))))
edges_undirected.cache()
edges_undirected.count()

# Initialize ranks uniformly
ranks_default = vertices.select("id").withColumn("rank", F.lit(1.0 / n_vertices))
ranks_default.cache()
ranks_default.count()

# Out-degree per source vertex
out_degree = edges_undirected.groupBy("src").count().withColumnRenamed("count", "out_degree")
out_degree.cache()
out_degree.count()

metrics_run1 = []
prev_ranks = ranks_default

for i in range(NUM_ITER):
    t0 = time.time()
    contribs = (
        edges_undirected.join(prev_ranks, edges_undirected.src == prev_ranks.id)
        .join(out_degree, "src")
        .select(F.col("dst").alias("id"),
                (F.col("rank") / F.col("out_degree")).alias("contrib"))
    )
    new_ranks = (
        contribs.groupBy("id")
        .agg(F.sum("contrib").alias("sum_contrib"))
        .withColumn("rank", F.lit((1 - DAMPING) / n_vertices) + F.lit(DAMPING) * F.col("sum_contrib"))
        .select("id", "rank")
    )
    new_ranks.cache()
    new_ranks.count()  # force materialization
    elapsed_ms = (time.time() - t0) * 1000

    # convergence: max delta
    delta_row = (new_ranks.alias("n")
                 .join(prev_ranks.alias("p"), "id")
                 .agg(F.max(F.abs(F.col("n.rank") - F.col("p.rank"))).alias("max_delta"))
                 .first())
    max_delta = float(delta_row["max_delta"]) if delta_row else 0.0

    metrics_run1.append({"iteration": i, "elapsed_ms": elapsed_ms, "max_delta": max_delta})
    print(f"[Run1 default] Iter {i}: {elapsed_ms:.0f} ms, max_delta={max_delta:.6f}")
    prev_ranks.unpersist()
    prev_ranks = new_ranks

ranks_final_default = prev_ranks
print("\nTop 10 heroes by PageRank (default partitioning):")
ranks_final_default.orderBy(F.desc("rank")).show(10)

[Run1 default] Iter 0: 573 ms, max_delta=0.028153
[Run1 default] Iter 1: 485 ms, max_delta=0.010263
[Run1 default] Iter 2: 325 ms, max_delta=0.002406
[Run1 default] Iter 3: 258 ms, max_delta=0.000704
[Run1 default] Iter 4: 276 ms, max_delta=0.000192
[Run1 default] Iter 5: 263 ms, max_delta=0.000055
[Run1 default] Iter 6: 268 ms, max_delta=0.000015
[Run1 default] Iter 7: 244 ms, max_delta=0.000004
[Run1 default] Iter 8: 306 ms, max_delta=0.000001
[Run1 default] Iter 9: 274 ms, max_delta=0.000000

Top 10 heroes by PageRank (default partitioning):
+-------+--------------------+
|     id|                rank|
+-------+--------------------+
|hero_01|0.039741313353736304|
|hero_02| 0.03877577467559649|
|hero_03| 0.03635993838594008|
|hero_04| 0.03626868652070423|
|hero_05| 0.03266316119954227|
|hero_06| 0.03148293269503196|
|hero_09| 0.03076155361296139|
|hero_10|0.030230783674795623|
|hero_07| 0.02993829238059835|
|hero_27|0.028090077362507297|
+-------+--------------------+
only showing to

## A.2bis — Save plan_iteration.txt and plan_before.txt

In [5]:
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)

# Plan of one iteration (the join+agg pattern)
demo_iter = (edges_undirected.join(ranks_final_default, edges_undirected.src == ranks_final_default.id)
             .join(out_degree, "src")
             .select(F.col("dst").alias("id"), (F.col("rank") / F.col("out_degree")).alias("contrib"))
             .groupBy("id").agg(F.sum("contrib").alias("sum_contrib")))

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    demo_iter.explain("formatted")
with open("proof/plan_iteration.txt", "w") as f:
    f.write(buf.getvalue())

with open("proof/plan_before.txt", "w") as f:
    f.write(buf.getvalue())
print("plan_iteration.txt and plan_before.txt saved.")

plan_iteration.txt and plan_before.txt saved.


## A.3 — RUN 2 (repartitioned edges by `src`)

In [6]:
# Re-partition edges by src to co-locate joins on src
edges_repart = edges_undirected.repartition(8, "src")
edges_repart.cache()
edges_repart.count()

out_degree_repart = edges_repart.groupBy("src").count().withColumnRenamed("count", "out_degree")
out_degree_repart.cache()
out_degree_repart.count()

ranks_repart = vertices.select("id").repartition(8, "id").withColumn("rank", F.lit(1.0 / n_vertices))
ranks_repart.cache()
ranks_repart.count()

metrics_run2 = []
prev_ranks = ranks_repart

for i in range(NUM_ITER):
    t0 = time.time()
    contribs = (
        edges_repart.join(prev_ranks, edges_repart.src == prev_ranks.id)
        .join(out_degree_repart, "src")
        .select(F.col("dst").alias("id"),
                (F.col("rank") / F.col("out_degree")).alias("contrib"))
    )
    new_ranks = (
        contribs.groupBy("id")
        .agg(F.sum("contrib").alias("sum_contrib"))
        .withColumn("rank", F.lit((1 - DAMPING) / n_vertices) + F.lit(DAMPING) * F.col("sum_contrib"))
        .select("id", "rank")
    )
    new_ranks.cache()
    new_ranks.count()
    elapsed_ms = (time.time() - t0) * 1000

    delta_row = (new_ranks.alias("n")
                 .join(prev_ranks.alias("p"), "id")
                 .agg(F.max(F.abs(F.col("n.rank") - F.col("p.rank"))).alias("max_delta"))
                 .first())
    max_delta = float(delta_row["max_delta"]) if delta_row else 0.0

    metrics_run2.append({"iteration": i, "elapsed_ms": elapsed_ms, "max_delta": max_delta})
    print(f"[Run2 repart] Iter {i}: {elapsed_ms:.0f} ms, max_delta={max_delta:.6f}")
    prev_ranks.unpersist()
    prev_ranks = new_ranks

ranks_final_repart = prev_ranks
print("\nTop 10 heroes by PageRank (repartitioned):")
ranks_final_repart.orderBy(F.desc("rank")).show(10)

[Run2 repart] Iter 0: 529 ms, max_delta=0.028153
[Run2 repart] Iter 1: 311 ms, max_delta=0.010263
[Run2 repart] Iter 2: 321 ms, max_delta=0.002406
[Run2 repart] Iter 3: 237 ms, max_delta=0.000704
[Run2 repart] Iter 4: 393 ms, max_delta=0.000192
[Run2 repart] Iter 5: 331 ms, max_delta=0.000055
[Run2 repart] Iter 6: 242 ms, max_delta=0.000015
[Run2 repart] Iter 7: 223 ms, max_delta=0.000004
[Run2 repart] Iter 8: 283 ms, max_delta=0.000001
[Run2 repart] Iter 9: 252 ms, max_delta=0.000000

Top 10 heroes by PageRank (repartitioned):
+-------+--------------------+
|     id|                rank|
+-------+--------------------+
|hero_01| 0.03974131335373631|
|hero_02| 0.03877577467559649|
|hero_03| 0.03635993838594008|
|hero_04|0.036268686520704224|
|hero_05| 0.03266316119954227|
|hero_06| 0.03148293269503196|
|hero_09| 0.03076155361296139|
|hero_10|0.030230783674795623|
|hero_07| 0.02993829238059835|
|hero_27|0.028090077362507293|
+-------+--------------------+
only showing top 10 rows


## A.3bis — Save plan_after.txt

In [7]:
demo_iter_after = (edges_repart.join(ranks_final_repart, edges_repart.src == ranks_final_repart.id)
                   .join(out_degree_repart, "src")
                   .select(F.col("dst").alias("id"), (F.col("rank") / F.col("out_degree")).alias("contrib"))
                   .groupBy("id").agg(F.sum("contrib").alias("sum_contrib")))

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    demo_iter_after.explain("formatted")
with open("proof/plan_after.txt", "w") as f:
    f.write(buf.getvalue())
print("plan_after.txt saved.")

plan_after.txt saved.


## A.4 — Save final ranks to Parquet

In [8]:
for path in ["outputs/lab3/ranks_default", "outputs/lab3/ranks_repart"]:
    if pathlib.Path(path).exists():
        shutil.rmtree(path)
pathlib.Path("outputs/lab3").mkdir(parents=True, exist_ok=True)

ranks_final_default.write.mode("overwrite").parquet("outputs/lab3/ranks_default")
ranks_final_repart.write.mode("overwrite").parquet("outputs/lab3/ranks_repart")
print("Ranks written to outputs/lab3/")

[Stage 401:>                                                        (0 + 8) / 8]

Ranks written to outputs/lab3/


## A.5 — Build `lab3_metrics_log.csv`

In [9]:
header = ["run_id", "algorithm", "task", "notes", "iteration",
          "convergence_metric", "shuffle_read_bytes_ui", "shuffle_write_bytes_ui",
          "elapsed_ms", "timestamp"]

rows = []
now = datetime.now().isoformat()

for m in metrics_run1:
    rows.append([1, "PageRank", f"pagerank_iter_{m['iteration']}_default",
                 "default partitioning - no repartition", m['iteration'],
                 round(m['max_delta'], 6), "", "", round(m['elapsed_ms'], 1), now])

for m in metrics_run2:
    rows.append([2, "PageRank", f"pagerank_iter_{m['iteration']}_repart",
                 "repartitioned by src (8 partitions)", m['iteration'],
                 round(m['max_delta'], 6), "", "", round(m['elapsed_ms'], 1), now])

with open("lab3_metrics_log.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(header)
    w.writerows(rows)

print("lab3_metrics_log.csv written:")
for r in rows:
    print(r)

lab3_metrics_log.csv written:
[1, 'PageRank', 'pagerank_iter_0_default', 'default partitioning - no repartition', 0, 0.028153, '', '', 572.8, '2026-05-10T13:17:20.737625']
[1, 'PageRank', 'pagerank_iter_1_default', 'default partitioning - no repartition', 1, 0.010263, '', '', 484.7, '2026-05-10T13:17:20.737625']
[1, 'PageRank', 'pagerank_iter_2_default', 'default partitioning - no repartition', 2, 0.002406, '', '', 324.8, '2026-05-10T13:17:20.737625']
[1, 'PageRank', 'pagerank_iter_3_default', 'default partitioning - no repartition', 3, 0.000704, '', '', 258.0, '2026-05-10T13:17:20.737625']
[1, 'PageRank', 'pagerank_iter_4_default', 'default partitioning - no repartition', 4, 0.000192, '', '', 275.9, '2026-05-10T13:17:20.737625']
[1, 'PageRank', 'pagerank_iter_5_default', 'default partitioning - no repartition', 5, 5.5e-05, '', '', 262.6, '2026-05-10T13:17:20.737625']
[1, 'PageRank', 'pagerank_iter_6_default', 'default partitioning - no repartition', 6, 1.5e-05, '', '', 268.3, '2026-05

## A.6 — Summary stats

In [10]:
total1 = sum(m['elapsed_ms'] for m in metrics_run1)
total2 = sum(m['elapsed_ms'] for m in metrics_run2)
print(f"Run 1 (default)  total elapsed: {total1:.0f} ms over {NUM_ITER} iterations")
print(f"Run 2 (repart)   total elapsed: {total2:.0f} ms over {NUM_ITER} iterations")
print(f"Speedup: {total1/total2:.2f}x" if total2 > 0 else "")
print(f"Final convergence delta Run 1: {metrics_run1[-1]['max_delta']:.6f}")
print(f"Final convergence delta Run 2: {metrics_run2[-1]['max_delta']:.6f}")

Run 1 (default)  total elapsed: 3271 ms over 10 iterations
Run 2 (repart)   total elapsed: 3123 ms over 10 iterations
Speedup: 1.05x
Final convergence delta Run 1: 0.000000
Final convergence delta Run 2: 0.000000


In [11]:
spark.stop()
print("Lab 3 complete.")

Lab 3 complete.
